In [1]:
from datasets import load_dataset, load_from_disk
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification, TrainingArguments, Trainer
import torch
import numpy as np
import evaluate
import matplotlib.pyplot as plt
from torchvision.transforms import RandomResizedCrop, Compose, Normalize, ToTensor

c:\Users\dzky17\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\dzky17\AppData\Local\Programs\Python\Python313\Lib\site-packages\transformers\utils\hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
data = load_dataset("imagefolder", data_dir="../data-set/clfdata/")

train_dataset = data["train"].shuffle(seed=42).select(range(1000))
eval_dataset = data["test"].shuffle(seed=42).select(range(200))
print(f"Jumlah data training: {len(train_dataset)}")
print(f"Jumlah data testing: {len(eval_dataset)}")

Jumlah data training: 1000
Jumlah data testing: 200


In [3]:
labels = data["train"].features["label"].names
id2label = {idx: label for idx, label in enumerate(labels)}
label2id = {label: idx for idx, label in enumerate(labels)}
num_labels = len(labels)

# Ganti checkpoint di sini dengan model yang ingin Anda uji
checkpoint = "google/mobilenet_v2_1.0_224"
model = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

Some weights of MobileNetV2ForImageClassification were not initialized from the model checkpoint at google/mobilenet_v2_1.0_224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1001]) in the checkpoint and torch.Size([15]) in the model instantiated
- classifier.weight: found shape torch.Size([1001, 1280]) in the checkpoint and torch.Size([15, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
image_processor = AutoImageProcessor.from_pretrained(checkpoint)
normalize = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)
size = (
    image_processor.size["shortest_edge"]
    if "shortest_edge" in image_processor.size
    else (image_processor.size["height"], image_processor.size["width"])
)
_transforms = Compose([RandomResizedCrop(size), ToTensor(), normalize])

def transforms(examples):
    examples["pixel_values"] = [_transforms(img.convert("RGB")) for img in examples["image"]]
    del examples["image"]
    return examples

train_dataset = train_dataset.with_transform(transforms)
eval_dataset = eval_dataset.with_transform(transforms)

Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 11397.57it/s]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [5]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

In [6]:
model_name = checkpoint.split("/")[-1]
args = TrainingArguments(
   output_dir="mobilenetv2-clfdata",
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=20, # Ubah jumlah epoch sesuai kebutuhan
    warmup_ratio=0.1,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    report_to="none",
    seed=42,
    fp16=True
)

In [7]:
trainer = Trainer(
    model,
    args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

In [8]:
trainer.train()

c:\Users\dzky17\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,2.791900,2.783644,0.070000
2,2.701600,2.701108,0.145000
3,2.582800,2.563236,0.180000
4,2.388300,2.448154,0.255000
5,2.259000,2.360247,0.290000
6,2.165100,2.186988,0.355000
7,2.017100,2.137888,0.385000
8,1.941900,2.099246,0.380000
9,1.824000,1.989990,0.465000
10,1.717500,1.957760,0.445000


c:\Users\dzky17\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\dzky17\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\dzky17\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\dzky17\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  

TrainOutput(global_step=320, training_loss=1.874734139442444, metrics={'train_runtime': 1664.7996, 'train_samples_per_second': 12.013, 'train_steps_per_second': 0.192, 'total_flos': 5.292106186752e+16, 'train_loss': 1.874734139442444, 'epoch': 20.0})

In [9]:
save_directory = "./model/classification/mobilenetv2-clfdata"
trainer.save_model(save_directory)

print(f"Model saved to: {save_directory}")  

Model saved to: ./model/classification/mobilenetv2-clfdata
